# OPTIMIZED AQI PREDICTION - Target: 13-16% MAPE Reduction
## Key Improvements:
1. Weather feature integration (temp, humidity, pressure)
2. Advanced feature engineering & scaling
3. Feature interactions & polynomial features
4. Stacking ensemble model
5. LightGBM (faster, better performance)
6. Improved hyperparameter tuning

## 1. Imports

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
import lightgbm as lgb

print('✓ All imports successful')

✓ All imports successful


## 2. Load Data

In [9]:
df = pd.read_csv('sensor12.csv', parse_dates=['date'])
df.rename(columns={'12': 'y', 'date': 'ds'}, inplace=True)
df = df.sort_values('ds').reset_index(drop=True)
df['y'] = pd.to_numeric(df['y'], errors='coerce')
df.dropna(subset=['y'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'✓ Shape: {df.shape}  |  {df["ds"].min().date()} -> {df["ds"].max().date()}')
print(f'✓ Features: {df.columns.tolist()}')

✓ Shape: (1177, 6)  |  2017-07-19 -> 2020-10-07
✓ Features: ['ds', 'y', 'temp', 'wind_speed', 'humidity', 'pressure']


## 3. ADVANCED Feature Engineering

In [15]:
def build_advanced_features(df):
    d = df[['ds', 'y', 'temp', 'wind_speed', 'humidity', 'pressure']].copy()
    
    # Temporal Lags
    for lag in [1, 2, 3, 7, 14, 30]:
        d[f'lag_{lag}'] = d['y'].shift(lag)
    
    y_past = d['y'].shift(1)
    
    # EWM
    for span in [3, 7, 14, 30]:
        d[f'ewm_{span}'] = y_past.ewm(span=span, adjust=False).mean()
    
    # Rolling Mean and Std
    for window in [3, 7, 14, 30]:
        d[f'rolling_mean_{window}'] = y_past.rolling(window).mean()
        d[f'rolling_std_{window}'] = y_past.rolling(window).std()
    
    # Weather Features - Lags
    for lag in [1, 3, 7]:
        d[f'temp_lag_{lag}'] = d['temp'].shift(lag)
        d[f'humidity_lag_{lag}'] = d['humidity'].shift(lag)
        d[f'wind_lag_{lag}'] = d['wind_speed'].shift(lag)
        d[f'pressure_lag_{lag}'] = d['pressure'].shift(lag)
    
    # Weather Rolling Averages
    for window in [7, 14]:
        d[f'temp_rolling_{window}'] = d['temp'].rolling(window).mean()
        d[f'humidity_rolling_{window}'] = d['humidity'].rolling(window).mean()
        d[f'wind_rolling_{window}'] = d['wind_speed'].rolling(window).mean()
    
    # Feature Interactions
    d['temp_humidity_interaction'] = d['temp'] * d['humidity']
    d['wind_pressure_interaction'] = d['wind_speed'] * d['pressure']
    d['temp_pressure_interaction'] = d['temp'] * d['pressure']
    
    # Temporal
    d['month'] = d['ds'].dt.month
    d['dayofweek'] = d['ds'].dt.dayofweek
    d['dayofyear'] = d['ds'].dt.dayofyear
    d['quarter'] = d['ds'].dt.quarter
    
    # Cyclical Encoding
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)
    d['dayofweek_sin'] = np.sin(2 * np.pi * d['dayofweek'] / 7)
    d['dayofweek_cos'] = np.cos(2 * np.pi * d['dayofweek'] / 7)
    d['dayofyear_sin'] = np.sin(2 * np.pi * d['dayofyear'] / 365)
    d['dayofyear_cos'] = np.cos(2 * np.pi * d['dayofyear'] / 365)
    
    # Volatility
    d['y_volatility_7'] = y_past.rolling(7).std()
    d['y_volatility_14'] = y_past.rolling(14).std()
    
    '''# Rate of Change
    d['y_diff_1'] = d['y'].diff(1)
    d['y_diff_7'] = d['y'].diff(7)'''
    
    # Expanding Mean
    d['y_expanding_mean'] = d['y'].expanding(min_periods=1).mean()
    
    return d

df_features = build_advanced_features(df)
df_features.dropna(inplace=True)
df_features.reset_index(drop=True, inplace=True)

print(f'✓ Total features created: {df_features.shape[1] - 2}')
print(f'✓ Feature shape: {df_features.shape}')

✓ Total features created: 56
✓ Feature shape: (1147, 58)


## 4. Train-Test Split & Scaling

In [16]:
feat_train = df_features[(df_features['ds'] >= '2017-08-01') & 
                         (df_features['ds'] < '2019-06-01')].reset_index(drop=True)

feat_test = df_features[(df_features['ds'] >= '2019-06-01') & 
                        (df_features['ds'] < '2019-11-01')].reset_index(drop=True)

EXCLUDE_COLS = ['ds', 'y', 'temp', 'humidity', 'wind_speed', 'pressure']
FEATURE_COLS = [col for col in df_features.columns if col not in EXCLUDE_COLS]

# Feature Scaling
scaler = StandardScaler()
X_train = feat_train[FEATURE_COLS].values
X_train_scaled = scaler.fit_transform(X_train)
y_train = feat_train['y'].values

X_test = feat_test[FEATURE_COLS].values
X_test_scaled = scaler.transform(X_test)
y_test = feat_test['y'].values

print(f'✓ Train shape: {feat_train.shape}')
print(f'✓ Test shape: {feat_test.shape}')
print(f'✓ Using {len(FEATURE_COLS)} features')
print(f'✓ Features scaled (mean=0, std=1)')

✓ Train shape: (652, 58)
✓ Test shape: (153, 58)
✓ Using 52 features
✓ Features scaled (mean=0, std=1)


## 5. Metrics Helper

In [17]:
def get_metrics(y_true, y_pred):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2 = r2_score(yt, yp)
    mape = mean_absolute_percentage_error(yt, yp) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape, 'R2': r2}

print('✓ Metrics function ready')

✓ Metrics function ready


## 6. LightGBM

In [18]:
lgb_params = {
    'num_leaves': [31, 50, 100],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [500, 1000],
    'min_data_in_leaf': [5, 10, 20],
    'feature_fraction': [0.7, 0.8, 0.9],
    'bagging_fraction': [0.7, 0.8, 0.9],
    'lambda_l1': [0, 0.1, 0.5],
    'lambda_l2': [0, 0.1, 0.5]
}

lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
lgb_search = RandomizedSearchCV(
    lgb_model, lgb_params, n_iter=15, cv=5, 
    scoring='neg_mean_absolute_percentage_error',
    random_state=42, n_jobs=-1, verbose=0
)

print('Training LightGBM...')
lgb_search.fit(X_train_scaled, y_train)
lgb_best = lgb_search.best_estimator_

lgb_pred = np.clip(lgb_best.predict(X_test_scaled), 0, None)
met_lgb = get_metrics(y_test, lgb_pred)

print(f'🎯 LightGBM Results: {met_lgb}')

Training LightGBM...
🎯 LightGBM Results: {'MAE': 5.382588649247857, 'RMSE': 7.6453345303476485, 'MAPE(%)': 22.682530225443273, 'R2': 0.19252583433298653}


## 7. XGBoost

In [14]:
xgb_params = {
    'n_estimators': [1000, 1500],
    'learning_rate': [0.005, 0.01, 0.03],
    'max_depth': [4, 5, 6],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0, 0.1],
    'reg_alpha': [0.01, 0.05, 0.1],
    'reg_lambda': [1.0, 1.5, 2.0]
}

xgb_model = xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1)
xgb_search = RandomizedSearchCV(xgb_model, xgb_params, n_iter=15, cv=5,
                                scoring='neg_mean_absolute_percentage_error',
                                random_state=42, n_jobs=-1, verbose=0)

print('Training XGBoost...')
xgb_search.fit(X_train_scaled, y_train)
xgb_best = xgb_search.best_estimator_

xgb_pred = np.clip(xgb_best.predict(X_test_scaled), 0, None)
met_xgb = get_metrics(y_test, xgb_pred)

print(f'🎯 XGBoost Results: {met_xgb}')

Training XGBoost...


KeyboardInterrupt: 

## 8. CatBoost

In [ ]:
cat_params = {
    'iterations': [1500, 2000],
    'learning_rate': [0.01, 0.03],
    'depth': [6, 8, 10],
    'l2_leaf_reg': [1, 3, 5],
    'bagging_temperature': [1, 2],
    'random_strength': [1, 2]
}

cat_model = CatBoostRegressor(loss_function='RMSE', random_seed=42, verbose=0)
cat_search = RandomizedSearchCV(cat_model, cat_params, n_iter=10, cv=5,
                                scoring='neg_mean_absolute_percentage_error',
                                random_state=42, n_jobs=-1, verbose=0)

print('Training CatBoost...')
cat_search.fit(X_train_scaled, y_train)
cat_best = cat_search.best_estimator_

cat_pred = np.clip(cat_best.predict(X_test_scaled), 0, None)
met_cat = get_metrics(y_test, cat_pred)

print(f'🎯 CatBoost Results: {met_cat}')

## 9. STACKING ENSEMBLE

In [ ]:
# Generate meta-features for ensemble
kf = KFold(n_splits=5, shuffle=True, random_state=42)
meta_features_train = np.zeros((X_train_scaled.shape[0], 3))
meta_features_test = np.zeros((X_test_scaled.shape[0], 3))

base_learners = [('lgb', lgb_best), ('xgb', xgb_best), ('cat', cat_best)]

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train = y_train[train_idx]
    
    for i, (name, model) in enumerate(base_learners):
        meta_features_train[val_idx, i] = np.clip(model.predict(X_fold_val), 0, None)
        meta_features_test[:, i] += np.clip(model.predict(X_test_scaled), 0, None) / 5

# Train meta-learner
meta_learner = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
meta_learner.fit(meta_features_train, y_train)

ensemble_pred = np.clip(meta_learner.predict(meta_features_test), 0, None)
met_ensemble = get_metrics(y_test, ensemble_pred)

print(f'🎯 STACKING ENSEMBLE Results: {met_ensemble}')

## 10. Model Comparison

In [ ]:
model_results = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost', 'STACKING ENSEMBLE'],
    'MAE': [met_lgb['MAE'], met_xgb['MAE'], met_cat['MAE'], met_ensemble['MAE']],
    'RMSE': [met_lgb['RMSE'], met_xgb['RMSE'], met_cat['RMSE'], met_ensemble['RMSE']],
    'MAPE(%)': [met_lgb['MAPE(%)'], met_xgb['MAPE(%)'], met_cat['MAPE(%)'], met_ensemble['MAPE(%)']],
    'R2': [met_lgb['R2'], met_xgb['R2'], met_cat['R2'], met_ensemble['R2']]
})

print('\n' + '='*80)
print('MODEL COMPARISON')
print('='*80)
print(model_results.to_string(index=False))
print('='*80)

best_mape_idx = model_results['MAPE(%)'].idxmin()
best_model = model_results.loc[best_mape_idx]
baseline_mape = 21.19
improvement = ((baseline_mape - best_model['MAPE(%)']) / baseline_mape) * 100

print(f'\n🏆 BEST MODEL: {best_model["Model"]}')
print(f'   MAPE: {best_model["MAPE(%)"]:>6.2f}%')
print(f'   📊 Improvement: {improvement:+.2f}% from baseline')

## 11. Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

ax = axes[0]
te_dates = pd.to_datetime(feat_test['ds'].values)

ax.plot(te_dates, y_test, 'o-', color='#2C3E50', lw=2.5, markersize=4, label='Actual', zorder=5)
ax.plot(te_dates, ensemble_pred, 's--', color='#E74C3C', lw=2, markersize=4, label='Ensemble', zorder=6)
ax.fill_between(te_dates, y_test - (y_test - ensemble_pred), y_test, alpha=0.2, color='#E74C3C')
ax.set_ylabel('AQI Value', fontsize=11)
ax.set_title('Optimized AQI Prediction: Stacking Ensemble vs Actual', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3, linestyle=':')

txt = (f"MAE={met_ensemble['MAE']:.2f}  RMSE={met_ensemble['RMSE']:.2f}  "
        f"MAPE={met_ensemble['MAPE(%)']:.2f}%  R²={met_ensemble['R2']:.3f}")
ax.text(0.01, 0.05, txt, transform=ax.transAxes, fontsize=10, 
        bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7), family='monospace')

ax2 = axes[1]
residuals = y_test - ensemble_pred
ax2.bar(te_dates, residuals, color=['green' if r > 0 else 'red' for r in residuals], alpha=0.6, width=0.8)
ax2.axhline(0, color='black', lw=1, linestyle='-')
ax2.set_ylabel('Residual', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.grid(alpha=0.2, linestyle=':')

plt.tight_layout()
plt.show()

print('✓ Visualization complete')

## 12. Summary

In [ ]:
print('\n' + '='*80)
print('OPTIMIZATION SUMMARY')
print('='*80)
print(f'\n📈 BASELINE (Original Models):')
print(f'   Best: GRU with MAPE = 21.19%')
print(f'\n🎯 TARGET:')
print(f'   Reduce MAPE by 13-16% → Target range: 17.8-18.4%')
print(f'\n✨ KEY IMPROVEMENTS IMPLEMENTED:')
print(f'   1. ✓ Weather feature integration (temp, humidity, pressure)')
print(f'   2. ✓ Advanced temporal features (cyclical encoding, interactions)')
print(f'   3. ✓ Feature scaling (StandardScaler for optimal convergence)')
print(f'   4. ✓ LightGBM (faster, better than GradientBoosting)')
print(f'   5. ✓ Hyperparameter optimization (RandomizedSearchCV)')
print(f'   6. ✓ Stacking Ensemble (Level-0: LGB+XGB+CAT, Level-1: Meta-learner)')
print(f'\n🏆 FINAL RESULTS:')
print(f'   Best Model: {best_model["Model"]}')
print(f'   MAPE: {best_model["MAPE(%)"]:>6.2f}%')
print(f'   Improvement: {improvement:>6.2f}% reduction from baseline')
if improvement >= 13:
    print(f'   ✓ TARGET ACHIEVED! ({improvement:.2f}% >= 13%)')
else:
    print(f'   ⚠ Target not fully met')
print(f'\n📊 Bonus Metrics:')
print(f'   R² Score: {best_model["R2"]:.4f}')
print(f'   MAE: {best_model["MAE"]:.2f}')
print(f'   RMSE: {best_model["RMSE"]:.2f}')
print('\n' + '='*80)